In [13]:
import pandas as pd
import numpy as np
import plotly.express as px

def ReadData(path):
    """
    Leemos los datos desde un archivo CSV y los cargamos en un DataFrame de pandas.

    Parameters:
    ruta de tipo string del csv a entrenar

    Returns:
    devuelve un datafream de pandas con los datos del csv
    """
    try:
        data = pd.read_csv(path)
        return data
    except Exception as e:
        print(f"Error al cargar los datos: {e}")
        return None

df = ReadData("datos/application_details_granted_2016.csv")
df.dtypes
df.columns
df.head()

,PUBLICATION_NUMBER,PUBLICATION_DATE,IPO_LOCATION,APPLICATION_TYPE_DESC,APPLICATION_NUMBER,DATE_OF_FILING,TITLE_OF_INVENTION,FIELD_OF_INVENTION,NO_OF_PAGES,NO_OF_CLAIMS,DATE_UPDATED_d_m_y
0,01/2016,01/01/2016,Delhi,CONVENTION APPLICATION,10/DEL/2010,04/01/2010,"""PRODUCTION OF A PHARMACEUTICAL CONTAINER FROM...",CHEMISTRY,25,24.0,19/04/2023
1,01/2016,01/01/2016,Mumbai,PCT NATIONAL PHASE APPLICATION,1005/MUMNP/2007,03/07/2007,“A MANUFACTURING METHOD OF AN ANODE PLATE OF A...,ELECTRICAL,21,12.0,19/04/2023
2,01/2016,01/01/2016,Kolkata,CONVENTION APPLICATION,104/KOL/2007,29/01/2007,MOTOR-DRIVEN SURGICAL CUTTING AND FASTENING IN...,BIOMEDICAL,39,10.0,19/04/2023
3,01/2016,01/01/2016,Mumbai,PCT NATIONAL PHASE APPLICATION,1045/MUMNP/2009,28/05/2009,SYSTEM AND METHOD FOR DIGITAL SIGNAL PROCESSING,ELECTRONICS,55,25.0,19/04/2023
4,01/2016,01/01/2016,Mumbai,PCT NATIONAL PHASE APPLICATION,1070/MUMNP/2009,03/06/2009,“STERILISATION INDICATOR”,PHYSICS,19,16.0,19/04/2023


Viendo por encima los datos podemos ver que tenemos:
TITLE_OF_INVENTION: título de la invencion.
FIELD_OF_INVENTION: campo tecnologico, como CHEMISTRY, ELECTRICAL, BIOMEDICAL, ELECTRONICS o PHYSICS.
NO_OF_PAGES: número de paginas.
NO_OF_CLAIMS: numero de reivindicaciones.
DATE_OF_FILING: fecha de presentacion de la solicitud.
PUBLICATION_DATE: fecha de publicacion.

Podemos empezar a buscar relaciones que hay entre las diferentes columnas, como por ejemplo el nº de pagias y el nº de reivindicaciones con regresion lineal.
Pero vemos mas interesante hacer un modelo de ML para inferir el tipo de campo tecnologico que es una patente segun su Denominacion.

Empezamos a conocer nuestro dataset, obtenemos el tamaño y tipo 

In [ ]:
""" Dimensiones del dataframe """
print("Dimensiones del dataframe:"+ str(df.shape))
"""Tipos de datos de cada columna"""
print("Tipos de datos:")
print(df.dtypes)
px.histogram(df,x="FIELD_OF_INVENTION",title="Distribución de patentes por campo",labels={"FIELD_OF_INVENTION": "Campo tecnológico","count": "Número de patentes"}).update_layout(yaxis_title="Número de patentes").show()

Dimensiones del dataframe:(8064, 11)
Tipos de datos:
PUBLICATION_NUMBER           str
PUBLICATION_DATE             str
IPO_LOCATION                 str
APPLICATION_TYPE_DESC        str
APPLICATION_NUMBER           str
DATE_OF_FILING               str
TITLE_OF_INVENTION           str
FIELD_OF_INVENTION           str
NO_OF_PAGES                int64
NO_OF_CLAIMS             float64
DATE_UPDATED_d_m_y           str
dtype: object


Analizando las clasificaciones con Farma, tenemos un problema y es que si cogemos un set de pruebas para el entrenamiento, si no lo hacemos bien, peude que obtengamos daots donde solo tengamos una mayoria de farma ya que tiene un volumen superior al resto, y no solo farma tambien mecanica y electronica, las 3 jutnas suman casi la cantidad entera del set de datos.

Para nuestro modelo usaremos el aprendizaje supervisado, donde:
X => la variable que usaremos para la prediccion, en este caso el titulo de la patente.
y => la varaible que queremos predecir, en este caso la clasificacion de la patente.

como hemos observado hay 15 variables.

Empezamos a separar nuestro set de entrenamiento y nuestro set de pruebas, siempre teniendo encuenta que nuestro set de entrenamiento sera de mayor contenido.

In [38]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score

datos =df[['TITLE_OF_INVENTION', 'FIELD_OF_INVENTION']].dropna().copy()
X = datos["TITLE_OF_INVENTION"]
y = datos["FIELD_OF_INVENTION"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Comprobamos las cantidades en cada x,y train y test

In [39]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (6451,)
X_test: (1613,)
y_train: (6451,)
y_test: (1613,)


Para poder entrenar necesitamos converir texto en numero y para ello usaremos TF-IDF, esto lo que hace es obtener un vocabulario y tranformar en vector numerico los datos de entranmiento y con los datos de test solo transformamos a vector numerico.

In [40]:
vectorizador = TfidfVectorizer()
X_train_tfidf = vectorizador.fit_transform(X_train)

X_test_tfidf = vectorizador.transform(X_test)

Una vez que lo tenemos todo vectorizado y con patrones numericos podemos usar nuestro modelo de entrenamiento.

In [42]:
modelo = SGDClassifier(
    loss="log_loss",
    max_iter=2,
    tol=None,
    random_state=42
)

Ahora si empezamos en entrenamiento

In [43]:
modelo.fit(X_train_tfidf, y_train)
print("Épocas realizadas:", modelo.n_iter_)
print("Categorías aprendidas:", modelo.classes_)

Épocas realizadas: 2
Categorías aprendidas: ['BIOCHEMISTRY' 'BIOMEDICAL' 'BIOTECHNOLOGY' 'CHEMISTRY' 'CIVIL'
 'COMMUNICATION' 'COMPUTER' 'ELECTRICAL' 'ELECTRONICS' 'FOOD' 'MECHANICAL'
 'METALLURGY' 'PHYSICS' 'POLYMER' 'TEXTILE']


Vamos a verificar las predicciones, y las metricas

In [44]:
y_pred = modelo.predict(X_test_tfidf)

resultados = pd.DataFrame({
    "Titulo": X_test.to_numpy(),
    "Real": y_test.to_numpy(),
    "Prediccion": y_pred
})

resultados["Correcto"] = (
    resultados["Real"] == resultados["Prediccion"]
)

resultados.head()

,Titulo,Real,Prediccion,Correcto
0,METHOD AND APPARATUS FOR REDUCING EMISSIONS IN...,MECHANICAL,ELECTRONICS,False
1,"""DIKETO-PIPERAZINE AND PIPERIDINE DERIVATIVES ...",CHEMISTRY,CHEMISTRY,True
2,A METHOD OF SURFACE TREATING A MECHANICAL PART...,METALLURGY,CHEMISTRY,False
3,PROCESS FOR PRODUCING FUNCTIONALISED FIVE-MEMB...,CHEMISTRY,CHEMISTRY,True
4,"""IMPROVED PROCESS FOR THE ADDITION OF THIOLATE...",CHEMISTRY,CHEMISTRY,True


In [45]:
accuracy = accuracy_score(y_test, y_pred)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

print(f"Accuracy: {accuracy:.2%}")
print(f"F1 Macro: {f1:.4f}")

Accuracy: 64.91%
F1 Macro: 0.3189


Vamos a genrar los archivos que nos llevaremos

In [46]:
import joblib

joblib.dump(modelo, "modelo_patentes.joblib")
joblib.dump(vectorizador, "vectorizador_tfidf.joblib")

['vectorizador_tfidf.joblib']